In [27]:
import sys
import json
import pickle
import argparse
import logging
import warnings
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from IPython.display import display

# ML
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    mean_absolute_error, log_loss
)
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.dummy import DummyClassifier, DummyRegressor

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

In [28]:
BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "bigdata" / "processed"
ARTIFACTS_DIR = BASE_DIR / "artifacts" / "experiments"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS = [5, 20, 200]
N_SPLITS = 3
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

VAL_CUTOFF = "2022-01-01"
TEST_CUTOFF = "2023-01-01"

TICKER_SUBSET = 50

RUN_NN = True

# GPU check
try:
    import torch
    TORCH_AVAILABLE = True
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"PyTorch доступен. Устройство: {DEVICE}")
except ImportError:
    TORCH_AVAILABLE = False
    DEVICE = None
    logger.warning("PyTorch не установлен. LSTM/Transformer недоступны.")

2026-05-21 02:23:13,826 [INFO] PyTorch доступен. Устройство: cuda


## Выбор метрик

- **Бинарная классификация (рост/падение):** ROC‑AUC  
  *Почему:* Оценивает способность модели правильно ранжировать объекты — для нас важнее, чтобы вероятность роста была выше для действительно растущих акций, чем точное бинарное предсказание. AUC не зависит от выбранного порога, что позволяет потом настраивать его под торговую стратегию.

- **Регрессия (прогноз доходности):** RMSE  
  *Почему:* Сильнее штрафует крупные ошибки, чем MAE. Для трейдера ошибиться на 5% вместо 0.5% гораздо опаснее, поэтому RMSE лучше отражает практическую полезность. R² используем только для диагностики (показывает, лучше ли модель наивного прогноза).

In [29]:
def load_features():
    with open(PROCESSED_DIR / "feature_columns.txt") as f:
        return [line.strip() for line in f if line.strip()]

def temporal_split(df, feature_cols, target_col, cutoff_date=TEST_CUTOFF):
    """Корректный временной сплит по дате."""
    train_mask = df["date"] < cutoff_date
    test_mask = df["date"] >= cutoff_date
    X_train = df.loc[train_mask, feature_cols]
    X_test = df.loc[test_mask, feature_cols]
    y_train = df.loc[train_mask, target_col]
    y_test = df.loc[test_mask, target_col]
    valid_train = y_train.notna()
    valid_test = y_test.notna()
    return (X_train[valid_train], X_test[valid_test],
            y_train[valid_train], y_test[valid_test])

def temporal_val_split(df, feature_cols, target_col, val_cutoff=VAL_CUTOFF, test_cutoff=TEST_CUTOFF):
    """Корректный временной сплит на train / val / test по дате."""
    train_mask = df["date"] < val_cutoff
    val_mask = (df["date"] >= val_cutoff) & (df["date"] < test_cutoff)
    test_mask = df["date"] >= test_cutoff

    X_train = df.loc[train_mask, feature_cols]
    X_val = df.loc[val_mask, feature_cols]
    X_test = df.loc[test_mask, feature_cols]
    y_train = df.loc[train_mask, target_col]
    y_val = df.loc[val_mask, target_col]
    y_test = df.loc[test_mask, target_col]

    valid_train = y_train.notna()
    valid_val = y_val.notna()
    valid_test = y_test.notna()

    return (X_train[valid_train], X_val[valid_val], X_test[valid_test],
            y_train[valid_train], y_val[valid_val], y_test[valid_test])

In [30]:
def prepare_data_for_model(X_train, X_test, model_name, X_val=None):
    """
    Подготовка данных под конкретную модель.
    CatBoost - NaN как есть.
    Линейные - StandardScaler + median imputation.
    Остальные - median imputation.
    """
    if X_val is not None:
        if model_name == 'catboost':
            return X_train, X_test, X_val

        # Imputation
        imputer = SimpleImputer(strategy='median')
        X_train_imp = pd.DataFrame(
            imputer.fit_transform(X_train),
            columns=X_train.columns,
            index=X_train.index
        )
        X_test_imp = pd.DataFrame(
            imputer.transform(X_test),
            columns=X_test.columns,
            index=X_test.index
        )
        X_val_imp = pd.DataFrame(
            imputer.transform(X_val),
            columns=X_val.columns,
            index=X_val.index
        )

        if model_name in ('logistic', 'ridge'):
            scaler = StandardScaler()
            X_train_imp = pd.DataFrame(
                scaler.fit_transform(X_train_imp),
                columns=X_train_imp.columns,
                index=X_train_imp.index
            )
            X_test_imp = pd.DataFrame(
                scaler.transform(X_test_imp),
                columns=X_test_imp.columns,
                index=X_test_imp.index
            )
            X_val_imp = pd.DataFrame(
                scaler.transform(X_val_imp),
                columns=X_val_imp.columns,
                index=X_val_imp.index
            )
        return X_train_imp, X_test_imp, X_val_imp
    else:
        if model_name == 'catboost':
            return X_train, X_test

        # Imputation
        imputer = SimpleImputer(strategy='median')
        X_train_imp = pd.DataFrame(
            imputer.fit_transform(X_train),
            columns=X_train.columns,
            index=X_train.index
        )
        X_test_imp = pd.DataFrame(
            imputer.transform(X_test),
            columns=X_test.columns,
            index=X_test.index
        )

        if model_name in ('logistic', 'ridge'):
            scaler = StandardScaler()
            X_train_imp = pd.DataFrame(
                scaler.fit_transform(X_train_imp),
                columns=X_train_imp.columns,
                index=X_train_imp.index
            )
            X_test_imp = pd.DataFrame(
                scaler.transform(X_test_imp),
                columns=X_test_imp.columns,
                index=X_test_imp.index
            )
        return X_train_imp, X_test_imp

In [31]:
def get_gpu_params(model_name):
    try:
        import torch
        if torch.cuda.is_available():
            if model_name == 'catboost':
                return {'task_type': 'GPU', 'devices': '0'}
            elif model_name == 'lightgbm':
                return {'device': 'gpu'}
            elif model_name == 'xgboost':
                return {'device': 'gpu', 'gpu_platform_id': 0, 'gpu_device_id': 0}
    except ImportError:
        pass
    return {}


def show_and_save_fig(name, fig=None, exp_dir=None):
    """Сначала показываем в ноутбуке, потом сохраняем."""
    if fig is None:
        fig = plt.gcf()
    fig.tight_layout()
    display(fig)
    if exp_dir is not None:
        path = exp_dir / "figures" / name
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=150, bbox_inches='tight')
        logger.info(f"График сохранён: {path}")
    plt.close(fig)


def save_json(data, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False, default=str)


def save_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def make_exp_dirs(exp_name):
    """Создаёт ТОЛЬКО корневую папку эксперимента, без подпапок."""
    base = ARTIFACTS_DIR / exp_name
    base.mkdir(parents=True, exist_ok=True)
    return base

In [32]:
MODEL_CONFIGS = {
    'catboost': {
        'binary': {'random_seed': RANDOM_SEED, 'verbose': False, 'allow_writing_files': False},
        'return': {'random_seed': RANDOM_SEED, 'verbose': False, 'allow_writing_files': False}
    },
    'lightgbm': {
        'binary': {'random_state': RANDOM_SEED, 'verbose': -1},
        'return': {'random_state': RANDOM_SEED, 'verbose': -1}
    },
    'xgboost': {
        'binary': {'random_state': RANDOM_SEED, 'verbosity': 0},
        'return': {'random_state': RANDOM_SEED, 'verbosity': 0}
    },
    'random_forest': {
        'binary': {'random_state': RANDOM_SEED, 'n_jobs': -1}
    },
    'logistic': {
        'binary': {'random_state': RANDOM_SEED, 'n_jobs': -1, 'max_iter': 1000}
    },
    'ridge': {
        'return': {'random_state': RANDOM_SEED}
    }
}


def get_model(model_name, target_type, **kwargs):
    """Фабрика моделей с возможностью переопределения параметров."""
    config = MODEL_CONFIGS.get(model_name, {}).get(target_type)
    if config is None:
        raise ValueError(f"Нет конфига для {model_name}/{target_type}")
    config = {**config, **kwargs}
    gpu_params = get_gpu_params(model_name)
    config.update(gpu_params)
    if target_type == 'binary':
        if model_name == 'catboost':
            return CatBoostClassifier(**config)
        elif model_name == 'lightgbm':
            return LGBMClassifier(**config)
        elif model_name == 'xgboost':
            return XGBClassifier(**config)
        elif model_name == 'random_forest':
            return RandomForestClassifier(**config)
        elif model_name == 'logistic':
            return LogisticRegression(**config)
    else:
        if model_name == 'catboost':
            return CatBoostRegressor(**config)
        elif model_name == 'lightgbm':
            return LGBMRegressor(**config)
        elif model_name == 'xgboost':
            return XGBRegressor(**config)
        elif model_name == 'ridge':
            return Ridge(**config)
    raise ValueError(f"Unknown model {model_name} for {target_type}")

In [33]:
def evaluate_model(model, X_test, y_test, target_type):
    """Полная оценка модели."""
    if target_type == 'binary':
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = (y_proba >= 0.5).astype(int)
        return {
            'accuracy': round(accuracy_score(y_test, y_pred), 4),
            'precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
            'recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
            'f1': round(f1_score(y_test, y_pred, zero_division=0), 4),
            'auc': round(roc_auc_score(y_test, y_proba), 4),
            'logloss': round(log_loss(y_test, y_proba), 4),
            'predictions_proba': y_proba,
            'predictions': y_pred
        }
    else:
        y_pred = model.predict(X_test)
        return {
            'rmse': round(np.sqrt(mean_squared_error(y_test, y_pred)), 6),
            'mae': round(mean_absolute_error(y_test, y_pred), 6),
            'r2': round(r2_score(y_test, y_pred), 4),
            'predictions': y_pred
        }


def compute_baseline(target_type, y_train, y_test):
    if target_type == 'binary':
        dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
        dummy.fit(np.zeros((len(y_train), 1)), y_train)
        y_pred = dummy.predict(np.zeros((len(y_test), 1)))
        y_proba = dummy.predict_proba(np.zeros((len(y_test), 1)))[:, 1]
        return {
            'accuracy': round(accuracy_score(y_test, y_pred), 4),
            'auc': round(roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.5, 4)
        }
    else:
        dummy = DummyRegressor(strategy='mean')
        dummy.fit(np.zeros((len(y_train), 1)), y_train)
        y_pred = dummy.predict(np.zeros((len(y_test), 1)))
        return {
            'rmse': round(np.sqrt(mean_squared_error(y_test, y_pred)), 6),
            'r2': round(r2_score(y_test, y_pred), 4)
        }

In [34]:
def fit_model(model, model_name, X_train, y_train, X_val=None, y_val=None):
    """Обёртка обучения с early stopping для бустингов."""
    if model_name == 'catboost':
        train_pool = Pool(X_train, y_train)
        if X_val is not None and y_val is not None:
            val_pool = Pool(X_val, y_val)
            model.fit(train_pool, eval_set=val_pool, verbose=False, early_stopping_rounds=50)
        else:
            model.fit(train_pool, verbose=False)
    elif model_name in ('lightgbm', 'xgboost'):
        eval_set = [(X_val, y_val)] if X_val is not None and y_val is not None else None
        if eval_set is not None:
            if model_name == 'xgboost':
                model.fit(X_train, y_train, eval_set=eval_set, verbose=False)
            else:
                model.fit(X_train, y_train, eval_set=eval_set)
        else:
            model.fit(X_train, y_train)
    else:
        model.fit(X_train, y_train)
    return model

In [35]:
def tune_model_optuna(model_name, target_type, X_train, y_train, X_val, y_val, n_trials=5):
    """
    Быстрый подбор гиперпараметров с помощью Optuna.
    Возвращает словарь лучших параметров.
    """
    def objective(trial):
        if model_name == 'catboost':
            params = {
                'depth': trial.suggest_int('depth', 3, 8),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
                'iterations': trial.suggest_int('iterations', 300, 1000),

            }
            model = get_model(model_name, target_type, **params)
            model.fit(Pool(X_train, y_train), eval_set=Pool(X_val, y_val), verbose=False, early_stopping_rounds=50)
        elif model_name == 'lightgbm':
            params = {
                'num_leaves': trial.suggest_int('num_leaves', 10, 50),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
                'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            }
            model = get_model(model_name, target_type, **params)
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
        elif model_name == 'xgboost':
            params = {
                'max_depth': trial.suggest_int('max_depth', 3, 8),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            }
            model = get_model(model_name, target_type, **params)
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        elif model_name == 'random_forest':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 300),
                'max_depth': trial.suggest_int('max_depth', 5, 15),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            }
            model = get_model(model_name, target_type, **params)
            model.fit(X_train, y_train)
        elif model_name == 'logistic':
            params = {
                'C': trial.suggest_float('C', 0.01, 10.0, log=True),
            }
            model = get_model(model_name, target_type, **params)
            model.fit(X_train, y_train)
        elif model_name == 'ridge':
            params = {
                'alpha': trial.suggest_float('alpha', 0.01, 10.0, log=True),
            }
            model = get_model(model_name, target_type, **params)
            model.fit(X_train, y_train)
        else:
            return 0.0
        
        # Оценка на валидации
        if target_type == 'binary':
            y_proba = model.predict_proba(X_val)[:, 1]
            score = roc_auc_score(y_val, y_proba)
        else:
            y_pred = model.predict(X_val)
            score = -np.sqrt(mean_squared_error(y_val, y_pred))
        return score
    
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_SEED), pruner=MedianPruner(n_startup_trials=2))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best_params = study.best_params
    logger.info(f"Лучшие параметры для {model_name} ({target_type}): {best_params}")
    return best_params

## Exp0: Очистка данных и подвыборка тикеров

In [39]:
# %% Exp0: Подготовка данных
logger.info("=== ЭКСПЕРИМЕНТ 0: Подготовка данных ===")

# Загрузка сырых данных
df = pd.read_parquet(PROCESSED_DIR / "combined_features.parquet")
logger.info(f"Исходный датасет: {df.shape}, тикеров: {df['ticker'].nunique()}")

top_tickers = (
    df.groupby("ticker")["volume"]
    .mean()
    .sort_values(ascending=False)
    .head(TICKER_SUBSET)
    .index.tolist()
)
df = df[df["ticker"].isin(top_tickers)].copy()
logger.info(f"Подвыборка топ-{TICKER_SUBSET} тикеров: {top_tickers}")

exp0_dir = make_exp_dirs("exp0_data_prep")
clean_path = exp0_dir / "configs" / "cleaned_dataset.parquet"
df.to_parquet(clean_path, index=False)
logger.info(f"Exp0 завершён. Финальный shape: {df.shape}")

feature_cols = load_features()

df_clean = df.copy()
outliers_count = {}

for col in tqdm(feature_cols, desc="Обработка выбросов в фичах"):
    data = df_clean[col].dropna()
    if len(data) == 0:
        continue
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    if IQR == 0:
        continue  # константный признак — не трогаем
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    # Считаем исходное количество выбросов
    outliers = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    outliers_count[col] = outliers
    # Заменяем выбросы на границы
    df_clean[col] = df_clean[col].clip(lower, upper)

# Статистика после обработки
after_stats = df_clean[feature_cols].describe(percentiles=[0.01, 0.99]).T

# Выводим информацию о количестве исправленных выбросов
out_df = pd.Series(outliers_count).sort_values(ascending=False)
logger.info(f"Всего признаков с выбросами: {(out_df > 0).sum()}")
logger.info("Топ-10 признаков по числу выбросов до обработки:")
logger.info(out_df.head(10).to_string())

df = df_clean

2026-05-21 02:29:22,124 [INFO] === ЭКСПЕРИМЕНТ 0: Подготовка данных ===
2026-05-21 02:29:22,407 [INFO] Исходный датасет: (766539, 124), тикеров: 249
2026-05-21 02:29:22,517 [INFO] Подвыборка топ-50 тикеров: ['TRNFP', 'GMKN', 'UGLD', 'GLTR', 'IRAO', 'FEES', 'SBER', 'VTBR', 'CARM', 'GAZP', 'EUTR', 'MTLR', 'TATN', 'SPBE', 'SGZH', 'SBERP', 'RNFT', 'ROSN', 'LKOH', 'GAZT', 'VKCO', 'MAGN', 'RUAL', 'ALRS', 'CHMF', 'NVTK', 'ASTR', 'WUSH', 'MOEX', 'HYDR', 'POLY', 'GECO', 'SVCB', 'FLOT', 'TGKB', 'ETLN', 'YNDX', 'TGKN', 'TCSG', 'AFLT', 'UPRO', 'OZON', 'NLMK', 'TATNP', 'PIKK', 'RTKM', 'GAZC', 'SNGS', 'FIXP', 'SVET']
2026-05-21 02:29:23,764 [INFO] Exp0 завершён. Финальный shape: (154766, 124)


Обработка выбросов в фичах: 100%|██████████| 109/109 [00:00<00:00, 171.50it/s]


2026-05-21 02:29:25,317 [INFO] Всего признаков с выбросами: 76
2026-05-21 02:29:25,318 [INFO] Топ-10 признаков по числу выбросов до обработки:
2026-05-21 02:29:25,318 [INFO] macro_inflation_delta1m    42021
real_rate_delta1m          39766
macd_hist                  39667
macro_inflation_delta3m    39030
macd                       38786
macd_signal                38738
real_rate_delta3m          31283
usd_rub_vol_20d_delta3m    30253
m2_growth_3m_delta1m       28820
news_count                 28805


## Exp1: Сравнение моделей

In [37]:
# %% Exp1: Сравнение моделей
logger.info("\n=== ЭКСПЕРИМЕНТ 1: Сравнение моделей ===")

exp_dir = make_exp_dirs("exp1_model_comparison")
results = []

for target_type, target_col in [('binary', 'target_binary_20d'), ('return', 'target_return_20d')]:
    if target_col not in df.columns:
        logger.warning(f"{target_col} отсутствует, пропускаем")
        continue

    X_train_raw, X_val, X_test_raw, y_train, y_val, y_test = temporal_val_split(
        df, feature_cols, target_col
    )
    baseline = compute_baseline(target_type, y_train, y_test)

    models = ['catboost', 'lightgbm', 'xgboost']
    if target_type == 'binary':
        models.append('random_forest')
    models.append('logistic' if target_type == 'binary' else 'ridge')

    for model_name in models:
        logger.info(f"  {model_name} / {target_type}")
        try:
            X_train, X_test, X_val = prepare_data_for_model(
                X_train_raw, X_test_raw, model_name, X_val
            )

            best_params = tune_model_optuna(model_name, target_type, X_train, y_train, X_val, y_val, n_trials=10)

            model = get_model(model_name, target_type, **best_params)
            model = fit_model(model, model_name, X_train, y_train, X_val, y_val)
            metrics = evaluate_model(model, X_test, y_test, target_type)
            preds = metrics.pop('predictions_proba', metrics.pop('predictions'))

            # Сохранение модели
            if model_name == 'catboost':
                model.save_model(str(exp_dir / "models" / f"{model_name}_{target_type}.cbm"))
            else:
                with open(exp_dir / "models" / f"{model_name}_{target_type}.pkl", 'wb') as f:
                    pickle.dump(model, f)
            # Предсказания
            pred_df = pd.DataFrame({'y_true': y_test.values, 'y_pred': preds if isinstance(preds, np.ndarray) else preds})
            save_csv(pred_df, exp_dir / "predictions" / f"{model_name}_{target_type}.csv")

            result = {'experiment': 'exp1_model_comparison', 'model': model_name, 'type': target_type,
                      'horizon': 20, 'baseline': baseline, 'metrics': metrics}
            results.append(result)
            save_json(result, exp_dir / "metrics" / f"{model_name}_{target_type}.json")
            logger.info(f"    -> {metrics}")
        except Exception as e:
            logger.error(f"    Ошибка {model_name}: {e}")

# График
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
binary = [r for r in results if r['type'] == 'binary']
if binary:
    df_bin = pd.DataFrame(binary)
    df_bin['auc'] = df_bin['metrics'].apply(lambda x: x['auc'])
    df_bin['baseline_auc'] = df_bin['baseline'].apply(lambda x: x['auc'])
    x = np.arange(len(df_bin))
    axes[0].bar(x - 0.35/2, df_bin['auc'], 0.35, label='Model AUC', color='steelblue')
    axes[0].bar(x + 0.35/2, df_bin['baseline_auc'], 0.35, label='Baseline', color='coral')
    axes[0].set_xticks(x); axes[0].set_xticklabels(df_bin['model'], rotation=45)
    axes[0].set_title('Binary: AUC'); axes[0].legend(); axes[0].set_ylim(0,1)
reg = [r for r in results if r['type'] == 'return']
if reg:
    df_reg = pd.DataFrame(reg)
    df_reg['rmse'] = df_reg['metrics'].apply(lambda x: x['rmse'])
    df_reg['baseline_rmse'] = df_reg['baseline'].apply(lambda x: x['rmse'])
    x = np.arange(len(df_reg))
    axes[1].bar(x - 0.35/2, df_reg['rmse'], 0.35, label='Model RMSE', color='steelblue')
    axes[1].bar(x + 0.35/2, df_reg['baseline_rmse'], 0.35, label='Baseline', color='coral')
    axes[1].set_xticks(x); axes[1].set_xticklabels(df_reg['model'], rotation=45)
    axes[1].set_title('Regression: RMSE'); axes[1].legend()
show_and_save_fig("model_comparison.png", fig, exp_dir)
save_json(results, exp_dir / "metrics" / "all_results.json")
logger.info("Exp1 завершён")

2026-05-21 02:23:16,748 [INFO] 
=== ЭКСПЕРИМЕНТ 1: Сравнение моделей ===
2026-05-21 02:23:16,852 [INFO]   catboost / binary


[I 2026-05-21 02:23:16,868] A new study created in memory with name: no-name-d0d01181-b3bd-45e8-bdb9-8e889c351acf
[I 2026-05-21 02:23:17,716] Trial 0 finished with value: 0.5556367731493566 and parameters: {'depth': 5, 'learning_rate': 0.08927180304353628, 'l2_leaf_reg': 0.8471801418819978, 'iterations': 719}. Best is trial 0 with value: 0.5556367731493566.
[I 2026-05-21 02:23:18,188] Trial 1 finished with value: 0.5263997724632034 and parameters: {'depth': 3, 'learning_rate': 0.01432169828911152, 'l2_leaf_reg': 0.0017073967431528124, 'iterations': 907}. Best is trial 0 with value: 0.5556367731493566.
[I 2026-05-21 02:23:18,842] Trial 2 finished with value: 0.49729754864710685 and parameters: {'depth': 6, 'learning_rate': 0.051059032093947576, 'l2_leaf_reg': 0.0012087541473056963, 'iterations': 979}. Best is trial 0 with value: 0.5556367731493566.
[I 2026-05-21 02:23:19,569] Trial 3 finished with value: 0.4178915809216143 and parameters: {'depth': 7, 'learning_rate': 0.0163056873462214

2026-05-21 02:23:22,997 [INFO] Лучшие параметры для catboost (binary): {'depth': 5, 'learning_rate': 0.08927180304353628, 'l2_leaf_reg': 0.8471801418819978, 'iterations': 719}
2026-05-21 02:23:23,613 [INFO]     -> {'accuracy': 0.5876, 'precision': 0.5735, 'recall': 0.7815, 'f1': 0.6615, 'auc': 0.6466, 'logloss': 0.679}
2026-05-21 02:23:23,613 [INFO]   lightgbm / binary


[I 2026-05-21 02:23:24,753] A new study created in memory with name: no-name-960a27f9-586a-4518-8b91-14d40f8a08c2
[I 2026-05-21 02:23:26,182] Trial 0 finished with value: 0.5317880229387748 and parameters: {'num_leaves': 25, 'learning_rate': 0.08927180304353628, 'feature_fraction': 0.892797576724562, 'bagging_fraction': 0.8394633936788146, 'min_child_samples': 12}. Best is trial 0 with value: 0.5317880229387748.
[I 2026-05-21 02:23:27,395] Trial 1 finished with value: 0.5581091435520757 and parameters: {'num_leaves': 16, 'learning_rate': 0.011430983876313222, 'feature_fraction': 0.9464704583099741, 'bagging_fraction': 0.8404460046972835, 'min_child_samples': 37}. Best is trial 1 with value: 0.5581091435520757.
[I 2026-05-21 02:23:28,384] Trial 2 finished with value: 0.545137417728493 and parameters: {'num_leaves': 10, 'learning_rate': 0.09330606024425668, 'feature_fraction': 0.9329770563201687, 'bagging_fraction': 0.6849356442713105, 'min_child_samples': 13}. Best is trial 1 with value

2026-05-21 02:23:39,397 [INFO] Лучшие параметры для lightgbm (binary): {'num_leaves': 37, 'learning_rate': 0.020497980520950188, 'feature_fraction': 0.8080272084711243, 'bagging_fraction': 0.8186841117373118, 'min_child_samples': 13}
2026-05-21 02:23:41,196 [INFO]     -> {'accuracy': 0.5575, 'precision': 0.6521, 'recall': 0.304, 'f1': 0.4147, 'auc': 0.609, 'logloss': 0.6835}
2026-05-21 02:23:41,196 [INFO]   xgboost / binary


...

2026-05-21 02:26:46,235 [INFO] Лучшие параметры для xgboost (return): {'max_depth': 3, 'learning_rate': 0.011430983876313222, 'subsample': 0.9464704583099741, 'colsample_bytree': 0.8404460046972835, 'min_child_weight': 8}
2026-05-21 02:26:46,935 [INFO]     -> {'rmse': np.float64(0.126322), 'mae': 0.088492, 'r2': -0.2009}
2026-05-21 02:26:46,936 [INFO]   ridge / return


[I 2026-05-21 02:26:48,257] A new study created in memory with name: no-name-82532965-dc59-469c-b31d-73b026d4b7a4
[I 2026-05-21 02:26:48,381] Trial 0 finished with value: -0.22356954564685533 and parameters: {'alpha': 0.13292918943162169}. Best is trial 0 with value: -0.22356954564685533.
[I 2026-05-21 02:26:48,506] Trial 1 finished with value: -0.21101870688779262 and parameters: {'alpha': 7.114476009343421}. Best is trial 1 with value: -0.21101870688779262.
[I 2026-05-21 02:26:48,660] Trial 2 finished with value: -0.21765239063994427 and parameters: {'alpha': 1.5702970884055387}. Best is trial 1 with value: -0.21101870688779262.
[I 2026-05-21 02:26:48,780] Trial 3 finished with value: -0.22092862513445222 and parameters: {'alpha': 0.6251373574521749}. Best is trial 1 with value: -0.21101870688779262.
[I 2026-05-21 02:26:48,909] Trial 4 finished with value: -0.2242609231689936 and parameters: {'alpha': 0.02938027938703535}. Best is trial 1 with value: -0.21101870688779262.
[I 2026-05-

2026-05-21 02:26:49,520 [INFO] Лучшие параметры для ridge (return): {'alpha': 7.114476009343421}
2026-05-21 02:26:49,681 [INFO]     -> {'rmse': np.float64(0.138349), 'mae': 0.099744, 'r2': -0.4405}


<Figure size 1400x600 with 2 Axes>

2026-05-21 02:26:49,882 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\exp1_model_comparison\figures\model_comparison.png
2026-05-21 02:26:49,884 [INFO] Exp1 завершён


## Exp2: Абляция фичей

In [40]:
# %% Exp2: Абляция фичей
logger.info("\n=== ЭКСПЕРИМЕНТ 2: Абляция фичей ===")

exp_dir = make_exp_dirs("exp2_ablation")
results = []

# Группы фичей (без пересечений)
groups = {
    'technical_sma': [c for c in feature_cols if c.startswith('sma_') or c.startswith('ema_')],
    'technical_rsi': [c for c in feature_cols if c.startswith('rsi')],
    'technical_macd': [c for c in feature_cols if c.startswith('macd')],
    'technical_bb': [c for c in feature_cols if c.startswith('bb_')],
    'technical_atr': [c for c in feature_cols if c.startswith('atr')],
    'technical_stoch': [c for c in feature_cols if c.startswith('stoch') or c.startswith('williams') or c.startswith('cci')],
    'technical_obv': [c for c in feature_cols if c.startswith('obv')],
    'technical_momentum': [c for c in feature_cols if c.startswith('momentum') or c.startswith('roc_') or c.startswith('drawdown')],
    'technical_intraday': [c for c in feature_cols if c.startswith('intraday') or c.startswith('upper_shadow') or c.startswith('lower_shadow') or c.startswith('body_size')],
    'technical_signals': [c for c in feature_cols if c.startswith('sma_20_50_cross') or c.startswith('sma_50_200_cross') or c.startswith('price_above_sma') or c.startswith('macd_bullish')],
    'volume_raw': [c for c in feature_cols if c.startswith('vol_ma') or c.startswith('vol_ratio') or c.startswith('volume_lag') or c.startswith('volume_change')],
    'volatility': [c for c in feature_cols if c.startswith('volatility') or c.startswith('parkinson')],
    'macro': [c for c in feature_cols if c.startswith('macro_') or c.startswith('usd_rub') or c.startswith('m2_growth') or c.startswith('real_rate')],
    'news': [c for c in feature_cols if c.startswith('news_')],
    'temporal': [c for c in feature_cols if c.startswith('day_of') or c.startswith('month') or c.startswith('quarter') or c.startswith('week_of') or c.startswith('days_') or c.startswith('is_')],
    'market': [c for c in feature_cols if c.startswith('market_') or c.startswith('relative_')],
    'lags': [c for c in feature_cols if c.startswith('close_lag') or c.startswith('log_ret_lag')],
}
logger.info(f"Группы: { {k: len(v) for k,v in groups.items()} }")

for target_type, target_col in [('binary', 'target_binary_20d'), ('return', 'target_return_20d')]:
    if target_col not in df.columns:
        continue
    X_train_raw, X_test_raw, y_train, y_test = temporal_split(df, feature_cols, target_col)
    baseline = compute_baseline(target_type, y_train, y_test)

    # Full features
    X_train, X_test = prepare_data_for_model(X_train_raw, X_test_raw, 'catboost')
    model = get_model('catboost', target_type)
    model.fit(Pool(X_train, y_train), verbose=False)
    full_metrics = evaluate_model(model, X_test, y_test, target_type)
    full_metrics.pop('predictions_proba', None); full_metrics.pop('predictions', None)
    results.append({'variant': 'all_features', 'type': target_type, 'n_features': len(feature_cols), 'baseline': baseline, 'metrics': full_metrics})

    for group_name, group_cols in tqdm(groups.items(), desc=f"Ablation {target_type}"):
        remaining = [c for c in feature_cols if c not in group_cols]
        if len(remaining) < 10:
            continue
        X_train_g = X_train_raw[remaining]
        X_test_g = X_test_raw[remaining]
        X_train_g, X_test_g = prepare_data_for_model(X_train_g, X_test_g, 'catboost')
        model = get_model('catboost', target_type)
        model.fit(Pool(X_train_g, y_train), verbose=False)
        metrics = evaluate_model(model, X_test_g, y_test, target_type)
        metrics.pop('predictions_proba', None); metrics.pop('predictions', None)
        results.append({'variant': f'without_{group_name}', 'type': target_type, 'n_features': len(remaining), 'baseline': baseline, 'metrics': metrics})
        logger.info(f"  Без {group_name}: {metrics}")

# График
fig, axes = plt.subplots(1, 2, figsize=(14,6))
for idx, target_type in enumerate(['binary', 'return']):
    sub = [r for r in results if r['type'] == target_type]
    if not sub: continue
    variants = [r['variant'] for r in sub]
    if target_type == 'binary':
        values = [r['metrics']['auc'] for r in sub]
        baseline_val = sub[0]['baseline']['auc']
        ylabel = 'AUC'
    else:
        values = [r['metrics']['rmse'] for r in sub]
        baseline_val = sub[0]['baseline']['rmse']
        ylabel = 'RMSE'
    colors = ['green' if v=='all_features' else 'steelblue' for v in variants]
    axes[idx].barh(variants, values, color=colors)
    axes[idx].axvline(baseline_val, color='red', linestyle='--', label='baseline')
    axes[idx].set_title(f'{target_type.capitalize()}: Feature Ablation')
    axes[idx].set_xlabel(ylabel); axes[idx].legend()
show_and_save_fig("ablation.png", fig, exp_dir)
save_json(results, exp_dir / "metrics" / "ablation.json")
logger.info("Exp2 завершён")

2026-05-21 02:29:27,965 [INFO] 
=== ЭКСПЕРИМЕНТ 2: Абляция фичей ===
2026-05-21 02:29:27,967 [INFO] Группы: {'technical_sma': 7, 'technical_rsi': 1, 'technical_macd': 4, 'technical_bb': 3, 'technical_atr': 1, 'technical_stoch': 4, 'technical_obv': 1, 'technical_momentum': 3, 'technical_intraday': 4, 'technical_signals': 4, 'volume_raw': 6, 'volatility': 2, 'macro': 40, 'news': 3, 'temporal': 14, 'market': 3, 'lags': 6}


Ablation binary:   0%|          | 0/17 [00:00<?, ?it/s]

2026-05-21 02:29:42,742 [INFO]   Без technical_sma: {'accuracy': 0.54, 'precision': 0.6155, 'recall': 0.2878, 'f1': 0.3922, 'auc': 0.5238, 'logloss': 0.7381}


Ablation binary:   6%|▌         | 1/17 [00:07<01:54,  7.17s/it]

2026-05-21 02:29:49,910 [INFO]   Без technical_rsi: {'accuracy': 0.5479, 'precision': 0.6596, 'recall': 0.2549, 'f1': 0.3678, 'auc': 0.5388, 'logloss': 0.7402}


...

Ablation return:  94%|█████████▍| 16/17 [01:23<00:05,  5.22s/it]

2026-05-21 02:33:09,119 [INFO]   Без lags: {'rmse': np.float64(0.150414), 'mae': 0.109875, 'r2': -0.7027}


Ablation return: 100%|██████████| 17/17 [01:29<00:00,  5.24s/it]


<Figure size 1400x600 with 2 Axes>

2026-05-21 02:33:09,440 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\exp2_ablation\figures\ablation.png
2026-05-21 02:33:09,443 [INFO] Exp2 завершён


## Exp3: Отбор Top-K фичей

In [41]:
# %% Exp3: Top-K фичей
logger.info("\n=== ЭКСПЕРИМЕНТ 3: Top-K фичей ===")

exp_dir = make_exp_dirs("exp3_top_k")
results = []

for target_type, target_col in [('binary', 'target_binary_20d'), ('return', 'target_return_20d')]:
    if target_col not in df.columns:
        continue
    X_train_raw, X_test_raw, y_train, y_test = temporal_split(df, feature_cols, target_col)
    X_train, X_test = prepare_data_for_model(X_train_raw, X_test_raw, 'catboost')

    model = get_model('catboost', target_type)
    model.fit(Pool(X_train, y_train), verbose=False)

    importance = pd.DataFrame({'feature': feature_cols, 'importance': model.get_feature_importance()}).sort_values('importance', ascending=False)
    save_csv(importance, exp_dir / "predictions" / f"feature_importance_{target_type}.csv")

    fig, ax = plt.subplots(figsize=(10,12))
    top_20 = importance.head(20)
    ax.barh(top_20['feature'][::-1], top_20['importance'][::-1], color='teal')
    ax.set_title(f'Top-20 Feature Importance ({target_type})')
    show_and_save_fig(f"importance_{target_type}.png", fig, exp_dir)

    k_values = [10, 20, 30, 50, 100, len(feature_cols)]
    for k in tqdm(k_values, desc=f"Top-K {target_type}"):
        if k > len(feature_cols): continue
        top_k = importance.head(k)['feature'].tolist()
        X_train_k = X_train[top_k]; X_test_k = X_test[top_k]
        model_k = get_model('catboost', target_type)
        model_k.fit(Pool(X_train_k, y_train), verbose=False)
        metrics = evaluate_model(model_k, X_test_k, y_test, target_type)
        metrics.pop('predictions_proba', None); metrics.pop('predictions', None)
        results.append({'k': k, 'type': target_type, 'metrics': metrics})
        logger.info(f"  Top-{k}: {metrics}")

fig, axes = plt.subplots(1,2,figsize=(14,6))
for idx, target_type in enumerate(['binary','return']):
    sub = [r for r in results if r['type'] == target_type]
    if not sub: continue
    ks = [r['k'] for r in sub]
    values = [r['metrics']['auc'] if target_type=='binary' else r['metrics']['rmse'] for r in sub]
    ylabel = 'AUC' if target_type=='binary' else 'RMSE'
    axes[idx].plot(ks, values, marker='o', linewidth=2, color='steelblue')
    axes[idx].set_xlabel('Top-K Features'); axes[idx].set_ylabel(ylabel)
    axes[idx].set_title(f'{target_type.capitalize()}: Performance vs K'); axes[idx].grid(True, alpha=0.3)
show_and_save_fig("top_k.png", fig, exp_dir)
save_json(results, exp_dir / "metrics" / "top_k.json")
logger.info("Exp3 завершён")

2026-05-21 02:33:09,458 [INFO] 
=== ЭКСПЕРИМЕНТ 3: Top-K фичей ===


<Figure size 1000x1200 with 1 Axes>

2026-05-21 02:33:16,852 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\exp3_top_k\figures\importance_binary.png


Top-K binary:   0%|          | 0/6 [00:00<?, ?it/s]

2026-05-21 02:33:21,769 [INFO]   Top-10: {'accuracy': 0.466, 'precision': 0.4225, 'recall': 0.0967, 'f1': 0.1574, 'auc': 0.4296, 'logloss': 0.932}


...

Top-K return:  83%|████████▎ | 5/6 [00:20<00:04,  4.45s/it]

2026-05-21 02:34:24,734 [INFO]   Top-109: {'rmse': np.float64(0.159553), 'mae': 0.115517, 'r2': -0.9159}


Top-K return: 100%|██████████| 6/6 [00:26<00:00,  4.35s/it]


<Figure size 1400x600 with 2 Axes>

2026-05-21 02:34:24,927 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\exp3_top_k\figures\top_k.png
2026-05-21 02:34:24,927 [INFO] Exp3 завершён


## Exp4: Ансамбли

In [42]:
# %% Exp4: Ансамбли (среднее и взвешенное)
logger.info("\n=== ЭКСПЕРИМЕНТ 4: Ансамбли ===")

exp_dir = make_exp_dirs("exp4_ensembles")
results = []

for target_type, target_col in [('binary', 'target_binary_20d'), ('return', 'target_return_20d')]:
    if target_col not in df.columns:
        continue
    X_train_raw, X_test_raw, y_train, y_test = temporal_split(df, feature_cols, target_col)
    baseline = compute_baseline(target_type, y_train, y_test)

    # Валидационный сплит 85/15 для весов
    val_split_idx = int(0.85 * len(X_train_raw))
    X_tr, X_val = X_train_raw.iloc[:val_split_idx], X_train_raw.iloc[val_split_idx:]
    y_tr, y_val = y_train.iloc[:val_split_idx], y_train.iloc[val_split_idx:]

    models = {}
    predictions = {}
    val_preds = {}

    for name in ['catboost', 'lightgbm', 'xgboost']:
        X_train, X_test = prepare_data_for_model(X_train_raw, X_test_raw, name)
        X_tr_m, X_val_m = prepare_data_for_model(X_tr, X_val, name)

        model = get_model(name, target_type)
        if target_type == 'binary' and name == 'catboost':
            neg, pos = y_tr.value_counts().sort_index()
            model.set_params(class_weights=[1, neg/pos])

        model = fit_model(model, name, X_tr_m, y_tr, X_val_m, y_val)
        models[name] = model

        if target_type == 'binary':
            predictions[name] = model.predict_proba(X_test)[:, 1]
            val_preds[name] = model.predict_proba(X_val_m)[:, 1]
        else:
            predictions[name] = model.predict(X_test)
            val_preds[name] = model.predict(X_val_m)

    # 1. Среднее
    ensemble_pred = np.mean(list(predictions.values()), axis=0)
    if target_type == 'binary':
        y_pred = (ensemble_pred >= 0.5).astype(int)
        metrics_mean = {'accuracy': round(accuracy_score(y_test, y_pred), 4), 'auc': round(roc_auc_score(y_test, ensemble_pred), 4)}
    else:
        metrics_mean = {'rmse': round(np.sqrt(mean_squared_error(y_test, ensemble_pred)), 4),
                        'mae': round(mean_absolute_error(y_test, ensemble_pred), 4),
                        'r2': round(r2_score(y_test, ensemble_pred), 4)}
    results.append({'variant': 'mean', 'type': target_type, 'baseline': baseline, 'metrics': metrics_mean, 'weights': {k: 1/3 for k in predictions}})

    # 2. Взвешенное по валидационной метрике
    if target_type == 'binary':
        aucs = {k: roc_auc_score(y_val, v) for k, v in val_preds.items()}
        total = sum(aucs.values())
        weights = {k: v/total for k, v in aucs.items()}
        weighted_pred = sum(weights[k] * predictions[k] for k in predictions)
        y_pred_w = (weighted_pred >= 0.5).astype(int)
        metrics_weighted = {'accuracy': round(accuracy_score(y_test, y_pred_w), 4), 'auc': round(roc_auc_score(y_test, weighted_pred), 4)}
    else:
        rmses = {k: np.sqrt(mean_squared_error(y_val, v)) for k, v in val_preds.items()}
        inv_rmses = {k: 1/max(v, 1e-9) for k, v in rmses.items()}
        total = sum(inv_rmses.values())
        weights = {k: v/total for k, v in inv_rmses.items()}
        weighted_pred = sum(weights[k] * predictions[k] for k in predictions)
        metrics_weighted = {'rmse': round(np.sqrt(mean_squared_error(y_test, weighted_pred)), 4),
                            'mae': round(mean_absolute_error(y_test, weighted_pred), 4),
                            'r2': round(r2_score(y_test, weighted_pred), 4)}
    results.append({'variant': 'weighted', 'type': target_type, 'baseline': baseline, 'metrics': metrics_weighted, 'weights': weights})
    logger.info(f"  Ensemble mean: {metrics_mean}")
    logger.info(f"  Ensemble weighted: {metrics_weighted}")

    # Сохраняем предсказания
    pred_df = pd.DataFrame(predictions)
    pred_df['ensemble_mean'] = ensemble_pred
    pred_df['ensemble_weighted'] = weighted_pred
    pred_df['y_true'] = y_test.values
    save_csv(pred_df, exp_dir / "predictions" / f"ensemble_{target_type}.csv")

save_json(results, exp_dir / "metrics" / "ensemble.json")
logger.info("Exp4 завершён")

2026-05-21 02:34:24,948 [INFO] 
=== ЭКСПЕРИМЕНТ 4: Ансамбли ===
2026-05-21 02:34:36,026 [INFO]   Ensemble mean: {'accuracy': 0.4901, 'auc': 0.4774}
2026-05-21 02:34:36,028 [INFO]   Ensemble weighted: {'accuracy': 0.4904, 'auc': 0.4778}
2026-05-21 02:34:43,831 [INFO]   Ensemble mean: {'rmse': np.float64(0.1352), 'mae': 0.0961, 'r2': -0.3764}
2026-05-21 02:34:43,831 [INFO]   Ensemble weighted: {'rmse': np.float64(0.1342), 'mae': 0.0954, 'r2': -0.356}
2026-05-21 02:34:43,972 [INFO] Exp4 завершён


## Exp5: Квантильные предсказания

In [43]:
# %% Exp5: Квантильные предсказания (0.1, 0.5, 0.9)
logger.info("\n=== ЭКСПЕРИМЕНТ 5: Квантильные предсказания ===")

exp_dir = make_exp_dirs("exp5_quantiles")
target_col = 'target_return_20d'
if target_col not in df.columns:
    raise KeyError(f"{target_col} отсутствует в данных")

X_train_raw, X_test_raw, y_train, y_test = temporal_split(df, feature_cols, target_col)
X_train, X_test = prepare_data_for_model(X_train_raw, X_test_raw, 'catboost')

quantiles = [0.1, 0.5, 0.9]
predictions = {}
for q in tqdm(quantiles, desc="Quantiles"):
    model = CatBoostRegressor(loss_function=f'Quantile:alpha={q}', eval_metric='RMSE', random_seed=RANDOM_SEED, verbose=False)
    model.fit(Pool(X_train, y_train), verbose=False)
    preds = model.predict(X_test)
    predictions[f'q{int(q*100)}'] = preds
    model.save_model(str(exp_dir / "models" / f"quantile_{int(q*100)}.cbm"))

pred_df = pd.DataFrame(predictions)
pred_df['y_true'] = y_test.values
pred_df['in_interval'] = (pred_df['y_true'] >= pred_df['q10']) & (pred_df['y_true'] <= pred_df['q90'])
coverage = pred_df['in_interval'].mean()
pred_df['interval_width'] = pred_df['q90'] - pred_df['q10']

results = [{'coverage_q10_q90': round(coverage, 4), 'mean_interval_width': round(pred_df['interval_width'].mean(), 4), 'median_interval_width': round(pred_df['interval_width'].median(), 4)}]
save_csv(pred_df, exp_dir / "predictions" / "quantile_predictions.csv")
save_json(results, exp_dir / "metrics" / "quantile.json")

# График
fig, ax = plt.subplots(figsize=(12,6))
sample_idx = np.random.choice(len(pred_df), min(100, len(pred_df)), replace=False)
sample = pred_df.iloc[sample_idx].sort_values('y_true')
x = range(len(sample))
ax.fill_between(x, sample['q10'], sample['q90'], alpha=0.3, label='90% CI')
ax.plot(x, sample['q50'], 'b-', label='Median (q50)')
ax.scatter(x, sample['y_true'], c='red', s=10, label='True', alpha=0.6)
ax.set_title(f'Quantile Predictions (Coverage: {coverage:.2%})')
ax.set_xlabel('Sample'); ax.set_ylabel('Return'); ax.legend()
show_and_save_fig("quantile_intervals.png", fig, exp_dir)
logger.info("Exp5 завершён")

2026-05-21 02:34:43,988 [INFO] 
=== ЭКСПЕРИМЕНТ 5: Квантильные предсказания ===


Quantiles: 100%|██████████| 3/3 [01:06<00:00, 22.22s/it]


<Figure size 1200x600 with 1 Axes>

2026-05-21 02:35:51,022 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\exp5_quantiles\figures\quantile_intervals.png
2026-05-21 02:35:51,023 [INFO] Exp5 завершён


## Нейросети (Exp6/Exp7)

In [44]:
# %% Exp6: LSTM (без NaN)
logger.info("\n=== ЭКСПЕРИМЕНТ 6: LSTM ===")
if not TORCH_AVAILABLE:
    raise RuntimeError("PyTorch не установлен – пропустите Exp6")

exp_dir = make_exp_dirs("exp6_lstm")
SEQ_LEN = 5
results = []

def prepare_nn_data(df_sub, feature_cols, target_col, seq_len=5, cutoff="2023-01-01"):
    # Берём нужные колонки + дату и тикер для группировки
    data = df_sub[feature_cols + [target_col, 'date', 'ticker']].copy()
    # Заполняем пропуски внутри каждого тикера
    for c in feature_cols:
        data[c] = data.groupby('ticker')[c].transform(lambda x: x.ffill(limit=20).bfill(limit=5))
    # Удаляем строки, где целевая переменная NaN
    data = data.dropna(subset=[target_col])
    # Масштабируем фичи (на этом этапе могут остаться NaN, если в группе все значения NaN)
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(data[feature_cols])
    # Преобразуем обратно в DataFrame для удобства удаления NaN
    scaled_df = pd.DataFrame(scaled_features, columns=feature_cols, index=data.index)
    # Удаляем строки, где в масштабированных фичах есть NaN
    valid_idx = ~scaled_df.isna().any(axis=1)
    scaled_df = scaled_df[valid_idx]
    targets = data.loc[valid_idx, target_col].values
    dates = data.loc[valid_idx, 'date'].values
    # Теперь строим последовательности
    X_all, y_all, date_all = [], [], []
    for i in range(len(scaled_df) - seq_len):
        X_all.append(scaled_df.iloc[i:i+seq_len].values)
        y_all.append(targets[i+seq_len])
        date_all.append(dates[i+seq_len])
    X_all = np.array(X_all, dtype=np.float32)
    y_all = np.array(y_all, dtype=np.float32)
    date_all = np.array(date_all)
    # Временной сплит
    train_mask = date_all < np.datetime64(cutoff)
    test_mask = ~train_mask
    # Дополнительно проверяем, что в тесте есть оба класса (для бинарной задачи)
    return (X_all[train_mask], X_all[test_mask],
            y_all[train_mask], y_all[test_mask], scaler)

def train_lstm(X_train_seq, y_train_seq, X_test_seq, y_test_seq, target_type, input_dim, epochs=15, batch_size=64):
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    # Для бинарной классификации убедимся, что в y_train есть оба класса
    if target_type == 'binary':
        unique = np.unique(y_train_seq)
        if len(unique) < 2:
            logger.warning(f"В тренировочных данных только один класс: {unique}, обучение бессмысленно")
            return None, {'error': 'single_class'}, None
        criterion = nn.BCEWithLogitsLoss()
        output_dim = 1
    else:
        criterion = nn.MSELoss()
        output_dim = 1

    class LSTMModel(nn.Module):
        def __init__(self, input_dim, hidden_dim=32, num_layers=2, output_dim=1, dropout=0.2):
            super().__init__()
            self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
            self.fc = nn.Linear(hidden_dim, output_dim)
            self.dropout = nn.Dropout(dropout)
        def forward(self, x):
            lstm_out, _ = self.lstm(x)
            return self.fc(self.dropout(lstm_out[:, -1, :]))

    model = LSTMModel(input_dim, output_dim=output_dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    train_dataset = TensorDataset(torch.FloatTensor(X_train_seq).to(DEVICE), torch.FloatTensor(y_train_seq).to(DEVICE))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    split_idx = int(0.8 * len(X_train_seq))
    X_val_seq = torch.FloatTensor(X_train_seq[split_idx:]).to(DEVICE)
    y_val_seq = torch.FloatTensor(y_train_seq[split_idx:]).to(DEVICE)

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    patience = 10
    history = {'train_loss': [], 'val_loss': []}

    for epoch in tqdm(range(epochs), desc="Training LSTM"):
        model.train()
        train_losses = []
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_x).squeeze(), batch_y)
            if torch.isnan(loss):
                logger.warning(f"NaN loss на эпохе {epoch}, прерывание")
                break
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_seq).squeeze(), y_val_seq).item()
        history['train_loss'].append(np.mean(train_losses))
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= patience:
            break

    if best_state is None:
        best_state = model.state_dict().copy()

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_outputs = model(torch.FloatTensor(X_test_seq).to(DEVICE)).squeeze().cpu().numpy()

    if target_type == 'binary':
        y_proba = 1 / (1 + np.exp(-test_outputs))
        # Проверка на NaN в предсказаниях
        if np.isnan(y_proba).any():
            logger.warning("Предсказания содержат NaN, возвращаем нулевые метрики")
            metrics = {'accuracy': 0.5, 'auc': 0.5, 'error': 'nan_predictions'}
        else:
            y_pred = (y_proba >= 0.5).astype(int)
            # Проверка, что в y_test есть оба класса
            if len(np.unique(y_test_seq)) < 2:
                auc_val = 0.5
            else:
                auc_val = roc_auc_score(y_test_seq, y_proba)
            metrics = {
                'accuracy': round(accuracy_score(y_test_seq, y_pred), 4),
                'auc': round(auc_val, 4)
            }
    else:
        # Регрессия
        if np.isnan(test_outputs).any():
            logger.warning("Предсказания содержат NaN, возвращаем нулевые метрики")
            metrics = {'rmse': 1e6, 'mae': 1e6, 'r2': -1e6, 'error': 'nan_predictions'}
        else:
            metrics = {
                'rmse': round(np.sqrt(mean_squared_error(y_test_seq, test_outputs)), 6),
                'mae': round(mean_absolute_error(y_test_seq, test_outputs), 6),
                'r2': round(r2_score(y_test_seq, test_outputs), 4)
            }
    return model, metrics, history

for target_type, target_col in [('binary', 'target_binary_20d'), ('return', 'target_return_20d')]:
    if target_col not in df.columns:
        continue
    X_train_seq, X_test_seq, y_train_seq, y_test_seq, scaler = prepare_nn_data(df, feature_cols, target_col, seq_len=SEQ_LEN)
    if len(X_train_seq) < 100 or len(X_test_seq) < 10:
        logger.warning(f"Недостаточно данных для LSTM ({len(X_train_seq)} train, {len(X_test_seq)} test)")
        continue
    model, metrics, history = train_lstm(X_train_seq, y_train_seq, X_test_seq, y_test_seq, target_type, input_dim=len(feature_cols), epochs=15, batch_size=64)
    if model is None:
        continue
    results.append({'type': target_type, 'seq_len': SEQ_LEN, 'metrics': metrics})
    save_json(results[-1], exp_dir / "metrics" / f"lstm_{target_type}.json")
    if history and 'train_loss' in history:
        fig, ax = plt.subplots(figsize=(10,5))
        ax.plot(history['train_loss'], label='Train Loss')
        ax.plot(history['val_loss'], label='Val Loss')
        ax.set_title(f'LSTM Training ({target_type})')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend()
        show_and_save_fig(f"lstm_{target_type}_history.png", fig, exp_dir)

logger.info("Exp6 завершён")

2026-05-21 02:35:51,048 [INFO] 
=== ЭКСПЕРИМЕНТ 6: LSTM ===


Training LSTM: 100%|██████████| 15/15 [01:40<00:00,  6.69s/it]


<Figure size 1000x500 with 1 Axes>

2026-05-21 02:37:36,961 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\exp6_lstm\figures\lstm_binary_history.png


Training LSTM: 100%|██████████| 15/15 [01:37<00:00,  6.53s/it]


<Figure size 1000x500 with 1 Axes>

2026-05-21 02:39:19,314 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\exp6_lstm\figures\lstm_return_history.png
2026-05-21 02:39:19,314 [INFO] Exp6 завершён


In [45]:
# %% Exp7: Transformer (без NaN)
logger.info("\n=== ЭКСПЕРИМЕНТ 7: Transformer ===")
if not TORCH_AVAILABLE:
    raise RuntimeError("PyTorch не установлен – пропустите Exp7")

exp_dir = make_exp_dirs("exp7_transformer")
SEQ_LEN = 5
results = []

def prepare_nn_data(df_sub, feature_cols, target_col, seq_len=5, cutoff="2023-01-01"):
    data = df_sub[feature_cols + [target_col, 'date', 'ticker']].copy()
    for c in feature_cols:
        data[c] = data.groupby('ticker')[c].transform(lambda x: x.ffill(limit=20).bfill(limit=5))
    data = data.dropna(subset=[target_col])
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(data[feature_cols])
    scaled_df = pd.DataFrame(scaled_features, columns=feature_cols, index=data.index)
    valid_idx = ~scaled_df.isna().any(axis=1)
    scaled_df = scaled_df[valid_idx]
    targets = data.loc[valid_idx, target_col].values
    dates = data.loc[valid_idx, 'date'].values
    X_all, y_all, date_all = [], [], []
    for i in range(len(scaled_df) - seq_len):
        X_all.append(scaled_df.iloc[i:i+seq_len].values)
        y_all.append(targets[i+seq_len])
        date_all.append(dates[i+seq_len])
    X_all = np.array(X_all, dtype=np.float32)
    y_all = np.array(y_all, dtype=np.float32)
    date_all = np.array(date_all)
    train_mask = date_all < np.datetime64(cutoff)
    test_mask = ~train_mask
    return (X_all[train_mask], X_all[test_mask],
            y_all[train_mask], y_all[test_mask], scaler)

def train_transformer(X_train_seq, y_train_seq, X_test_seq, y_test_seq, target_type, input_dim, epochs=15, batch_size=64):
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    if target_type == 'binary':
        unique = np.unique(y_train_seq)
        if len(unique) < 2:
            logger.warning(f"В тренировочных данных только один класс: {unique}, обучение бессмысленно")
            return None, {'error': 'single_class'}, None
        criterion = nn.BCEWithLogitsLoss()
        output_dim = 1
    else:
        criterion = nn.MSELoss()
        output_dim = 1

    class TransformerModel(nn.Module):
        def __init__(self, input_dim, d_model=32, nhead=4, num_layers=2, output_dim=1, dropout=0.2):
            super().__init__()
            self.embedding = nn.Linear(input_dim, d_model)
            encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True)
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            self.fc = nn.Linear(d_model, output_dim)
            self.dropout = nn.Dropout(dropout)
        def forward(self, x):
            x = self.embedding(x)
            x = self.transformer(x)
            return self.fc(self.dropout(x[:, -1, :]))

    model = TransformerModel(input_dim, output_dim=output_dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    train_dataset = TensorDataset(torch.FloatTensor(X_train_seq).to(DEVICE), torch.FloatTensor(y_train_seq).to(DEVICE))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    split_idx = int(0.8 * len(X_train_seq))
    X_val_seq = torch.FloatTensor(X_train_seq[split_idx:]).to(DEVICE)
    y_val_seq = torch.FloatTensor(y_train_seq[split_idx:]).to(DEVICE)

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    patience = 10
    history = {'train_loss': [], 'val_loss': []}

    for epoch in tqdm(range(epochs), desc="Training Transformer"):
        model.train()
        train_losses = []
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_x).squeeze(), batch_y)
            if torch.isnan(loss):
                logger.warning(f"NaN loss на эпохе {epoch}, прерывание")
                break
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_seq).squeeze(), y_val_seq).item()
        history['train_loss'].append(np.mean(train_losses))
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= patience:
            break

    if best_state is None:
        best_state = model.state_dict().copy()

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_outputs = model(torch.FloatTensor(X_test_seq).to(DEVICE)).squeeze().cpu().numpy()

    if target_type == 'binary':
        y_proba = 1 / (1 + np.exp(-test_outputs))
        if np.isnan(y_proba).any():
            logger.warning("Предсказания содержат NaN, возвращаем нулевые метрики")
            metrics = {'accuracy': 0.5, 'auc': 0.5, 'error': 'nan_predictions'}
        else:
            y_pred = (y_proba >= 0.5).astype(int)
            if len(np.unique(y_test_seq)) < 2:
                auc_val = 0.5
            else:
                auc_val = roc_auc_score(y_test_seq, y_proba)
            metrics = {
                'accuracy': round(accuracy_score(y_test_seq, y_pred), 4),
                'auc': round(auc_val, 4)
            }
    else:
        if np.isnan(test_outputs).any():
            logger.warning("Предсказания содержат NaN, возвращаем нулевые метрики")
            metrics = {'rmse': 1e6, 'mae': 1e6, 'r2': -1e6, 'error': 'nan_predictions'}
        else:
            metrics = {
                'rmse': round(np.sqrt(mean_squared_error(y_test_seq, test_outputs)), 6),
                'mae': round(mean_absolute_error(y_test_seq, test_outputs), 6),
                'r2': round(r2_score(y_test_seq, test_outputs), 4)
            }
    return model, metrics, history

for target_type, target_col in [('binary', 'target_binary_20d'), ('return', 'target_return_20d')]:
    if target_col not in df.columns:
        continue
    X_train_seq, X_test_seq, y_train_seq, y_test_seq, scaler = prepare_nn_data(df, feature_cols, target_col, seq_len=SEQ_LEN)
    if len(X_train_seq) < 100 or len(X_test_seq) < 10:
        logger.warning(f"Недостаточно данных для Transformer ({len(X_train_seq)} train, {len(X_test_seq)} test)")
        continue
    model, metrics, history = train_transformer(X_train_seq, y_train_seq, X_test_seq, y_test_seq, target_type, input_dim=len(feature_cols), epochs=15, batch_size=64)
    if model is None:
        continue
    results.append({'type': target_type, 'seq_len': SEQ_LEN, 'metrics': metrics})
    save_json(results[-1], exp_dir / "metrics" / f"transformer_{target_type}.json")
    if history and 'train_loss' in history:
        fig, ax = plt.subplots(figsize=(10,5))
        ax.plot(history['train_loss'], label='Train Loss')
        ax.plot(history['val_loss'], label='Val Loss')
        ax.set_title(f'Transformer Training ({target_type})')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend()
        show_and_save_fig(f"transformer_{target_type}_history.png", fig, exp_dir)

logger.info("Exp7 завершён")

2026-05-21 02:39:19,340 [INFO] 
=== ЭКСПЕРИМЕНТ 7: Transformer ===


Training Transformer: 100%|██████████| 15/15 [03:41<00:00, 14.77s/it]


<Figure size 1000x500 with 1 Axes>

2026-05-21 02:43:05,435 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\exp7_transformer\figures\transformer_binary_history.png


Training Transformer: 100%|██████████| 15/15 [03:37<00:00, 14.48s/it]


<Figure size 1000x500 with 1 Axes>

2026-05-21 02:46:46,834 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\exp7_transformer\figures\transformer_return_history.png
2026-05-21 02:46:46,835 [INFO] Exp7 завершён


## Exp8: Сравнение горизонтов

In [47]:
# %% Exp8: Сравнение горизонтов (5, 20, 200 дней)
logger.info("\n=== ЭКСПЕРИМЕНТ 8: Сравнение горизонтов ===")

exp_dir = make_exp_dirs("exp8_horizons")
results = []

for horizon in tqdm(HORIZONS, desc="Horizons"):
    for target_type, target_col in [('binary', f'target_binary_{horizon}d'), ('return', f'target_return_{horizon}d')]:
        if target_col not in df.columns:
            continue
        X_train_raw, X_test_raw, y_train, y_test = temporal_split(df, feature_cols, target_col)
        baseline = compute_baseline(target_type, y_train, y_test)

        X_train, X_test = prepare_data_for_model(X_train_raw, X_test_raw, 'catboost')
        model = get_model('catboost', target_type)
        model.fit(Pool(X_train, y_train), verbose=False)
        metrics = evaluate_model(model, X_test, y_test, target_type)
        metrics.pop('predictions_proba', None); metrics.pop('predictions', None)
        results.append({'horizon': horizon, 'type': target_type, 'baseline': baseline, 'metrics': metrics})
        logger.info(f"  h={horizon} {target_type}: {metrics}")

fig, axes = plt.subplots(1,2,figsize=(14,6))
for idx, target_type in enumerate(['binary','return']):
    sub = [r for r in results if r['type'] == target_type]
    if not sub: continue
    horizons = [r['horizon'] for r in sub]
    if target_type == 'binary':
        values = [r['metrics']['auc'] for r in sub]
        baseline_vals = [r['baseline']['auc'] for r in sub]
        ylabel = 'AUC'
    else:
        values = [r['metrics']['rmse'] for r in sub]
        baseline_vals = [r['baseline']['rmse'] for r in sub]
        ylabel = 'RMSE'
    axes[idx].plot(horizons, values, marker='o', linewidth=2, label='CatBoost', color='steelblue')
    axes[idx].plot(horizons, baseline_vals, marker='x', linewidth=2, label='Baseline', color='coral')
    axes[idx].set_xlabel('Horizon (days)'); axes[idx].set_ylabel(ylabel)
    axes[idx].set_title(f'{target_type.capitalize()}: Performance vs Horizon')
    axes[idx].set_xscale('log'); axes[idx].legend(); axes[idx].grid(True, alpha=0.3)
show_and_save_fig("horizon_comparison.png", fig, exp_dir)
save_json(results, exp_dir / "metrics" / "horizon.json")
logger.info("Exp8 завершён")

2026-05-21 02:55:43,256 [INFO] 
=== ЭКСПЕРИМЕНТ 8: Сравнение горизонтов ===


Horizons:   0%|          | 0/3 [00:00<?, ?it/s]Warning: less than 75% GPU memory available for training. Free: 1369.400001 Total: 4095.5


2026-05-21 02:55:50,600 [INFO]   h=5 binary: {'accuracy': 0.5303, 'precision': 0.5325, 'recall': 0.6047, 'f1': 0.5663, 'auc': 0.5397, 'logloss': 0.7148}


2026-05-21 02:55:55,868 [INFO]   h=5 return: {'rmse': np.float64(0.075985), 'mae': 0.047658, 'r2': -0.5141}


Horizons:  33%|███▎      | 1/3 [00:12<00:25, 12.61s/it]Warning: less than 75% GPU memory available for training. Free: 1369.400001 Total: 4095.5


...

2026-05-21 02:56:16,123 [INFO]   h=200 binary: {'accuracy': 0.7013, 'precision': 0.5366, 'recall': 0.6924, 'f1': 0.6047, 'auc': 0.7639, 'logloss': 0.6112}


2026-05-21 02:56:21,594 [INFO]   h=200 return: {'rmse': np.float64(0.345941), 'mae': 0.275202, 'r2': 0.1122}


Horizons: 100%|██████████| 3/3 [00:38<00:00, 12.78s/it]


<Figure size 1400x600 with 2 Axes>

2026-05-21 02:56:22,087 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\exp8_horizons\figures\horizon_comparison.png
2026-05-21 02:56:22,087 [INFO] Exp8 завершён
